<a href="https://colab.research.google.com/github/paplpap/zero-to-AI/blob/main/huggingface_tuto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hugging Face가 뭔가요?

한마디로 정의하자면 **"전 세계 인공지능 모델들의 '앱스토어'이자 '공용 도서관'"**입니다.

보통 인공지능 모델 하나를 만들려면 수억 원의 비용과 수개월의 시간이 듭니다. 그런데 구글, 메타(페이스북), 마이크로소프트 같은 공룡 기업들이 자신들이 만든 똑똑한 모델을 "누구나 공짜로 써보세요!"라며 허깅페이스에 올려두었습니다.

게다가, gpu를 가진 개인, 연구실등에서 각각의 목적에 특화되게 학습된(fine-tuning) 모델 또한 가져다 쓸 수 있습니다.

우리는 그저 그곳에 접속해서 내 업무에 필요한 모델을 **'쇼핑'**해서 가져오기만 하면 됩니다.

### Hugging Face Transformers 라이브러리 설치
Hugging Face의 모델들을 사용하려면 `transformers`라는 라이브러리가 필요합니다. 아래 코드를 실행해서 설치해봅시다!

In [1]:
!pip install transformers

### 인코더 모델 불러오기

이제 모델을 불러와볼까요? 우리는 **감성 분석**을 해주는 모델을 사용해볼 거예요. 감성 분석은 문장이 긍정적인지, 부정적인지를 판단해주는 기술입니다. 문과생 분들을 위해 한국어 감성 분석 모델을 사용해볼게요.

모델을 불러올 때는 단순히 모델 자체만 불러오는 것이 아니라, 모델이 이해할 수 있는 형태로 문장을 바꿔주는 '토크나이저(Tokenizer)'도 함께 불러와야 해요. 토크나이저는 문장을 단어나 의미 있는 조각으로 나누고 숫자로 변환해주는 역할을 합니다.

Pipeline의 추가적인 설명은 공식문서를 통해 확인하시기 바랍니다.
https://huggingface.co/docs/transformers/en/main_classes/pipelines#natural-language-processing

In [2]:
from transformers import pipeline
import torch


nlp_pipeline = pipeline(
    task="sentiment-analysis",
    model="daekeun-ml/koelectra-small-v3-nsmc",     # 사용할 모델
    device=0,                                       # 0은 GPU(있을 때), -1은 CPU
    torch_dtype=torch.float16,                      # 정밀도를 낮춰 메모리를 아끼고 속도를 높임 (반올림 같은 개념)
    num_workers=4,                                  # 데이터 읽어오는 일꾼 수 (CPU 코어 활용)
    model_kwargs={"low_cpu_mem_usage": True}        # 메모리 효율 최적화 옵션
)

# 분석할 데이터 (자신의 데이터로 바꿔보기)
reviews = ["정말 최고의 서비스였어요!", "다시는 이용하고 싶지 않네요.", "평범합니다."] * 10


results = nlp_pipeline(
    reviews,
    batch_size=16,           # 한 번에 16개씩 묶어서 처리
    truncation=True,         # 문장이 너무 길면 모델 한계치에 맞춰 자르기
    padding=True,            # 문장 길이를 맞춰서 계산 속도 향상
    top_k=None,              # 모든 라벨(긍정/부정)의 점수를 다 보고 싶을 때
    tokenizer_kwargs={"clean_up_tokenization_spaces": True} # 토큰화 후 공백 깔끔하게 정리
)

# 결과 확인
for i, res in enumerate(results[:3]):
    print(f"리뷰: {reviews[i]} -> 결과: {res}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/56.5M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


리뷰: 정말 최고의 서비스였어요! -> 결과: [{'label': '1', 'score': 0.9916519522666931}, {'label': '0', 'score': 0.008348053321242332}]
리뷰: 다시는 이용하고 싶지 않네요. -> 결과: [{'label': '0', 'score': 0.9988259673118591}, {'label': '1', 'score': 0.0011740822810679674}]
리뷰: 평범합니다. -> 결과: [{'label': '0', 'score': 0.9980770349502563}, {'label': '1', 'score': 0.0019229825120419264}]


daekeun-ml/koelectra-small-v3-nsmc 라는 모델을 가져와서 gpu사용, 양자화 사용등의 옵션을 킨 파이프 라인을 만들었습니다.

이후 가상의 예시 데이터를 넣어서 각 문장이 긍정적인 문장인지 아닌지를 숫자로 확인할 수 있었습니다.



---




*   리뷰: 정말 최고의 서비스였어요! -> 결과: [{'label': '1', 'score': 0.9916519522666931}, {'label': '0', 'score': 0.008348053321242332}]

*   리뷰: 다시는 이용하고 싶지 않네요. -> 결과: [{'label': '0', 'score': 0.9988259673118591}, {'label': '1', 'score': 0.0011740822810679674}]

*   리뷰: 평범합니다. -> 결과: [{'label': '0', 'score': 0.9980770349502563}, {'label': '1', 'score': 0.0019229825120419264}]



---



결과물을 보면 아시겠지만 label 1이 긍정 0이 부정입니다. 이러한 모델이 나타내는 값은 모델 상세 페이지에서 확인하시는게 좋습니다.

이러한 방식으로 기존의 리뷰 데이터를 이진 분류할 수 있습니다

※ 모델을 직접 허깅페이스에서 찾아 적용해 보고 학습한 데이터에 따른 차이를 관찰해 보십시오

### 디코더 모델 불러와서 사용하기

이제 디코더(Decoder) 모델을 실습해 보겠습니다. 이번 시간에 우리와 함께할 모델은 beomi/gemma-ko-2b로, 파라미터 수가 약 30억 개(3B)에 불과한 소형 모델입니다. 우리가 흔히 쓰는 ChatGPT나 Gemini처럼 거대한 모델들은 수천억 개의 파라미터를 통해 완성된 답변을 내놓지만, 이 작은 모델은 디코더 본연의 역할인 **'다음에 올 가장 확률 높은 단어 예측'**에 집중하는 모습을 보여줍니다. 비록 챗봇처럼 세련된 문장은 아닐지라도, 인공지능이 문장의 맥락을 어떻게 파악하고 단어를 이어가는지 그 원리를 이해하기에는 최적의 도구가 될 것입니다.

In [10]:
from transformers import pipeline, GenerationConfig
import torch

pipe = pipeline(
    "text-generation",
    model="beomi/gemma-ko-2b",
    device=0,
    torch_dtype=torch.float16
)

# 프롬프트 수정
# (유저 프롬프트는 우리가 사용하는 부분, 시스템 프롬프트는 모델에 전해주는 규칙을 말합니다)
user_question = "데이터 분석이 왜 재밌어? 문과생의 시선에서 3가지 이유를 들어줘."
prompt = f"<start_of_turn>user\n{user_question}<end_of_turn>\n<start_of_turn>model\n"


gen_config = GenerationConfig(
    max_new_tokens=512,
    do_sample=True,
    temperature=0.4,           # 높을수록 나올 확률이 낮은 단어가 나오게 됨
    top_p=0.85,                # 조금 더 핵심 단어에 집중하도록 낮춤
    repetition_penalty=1.5,    # 반복 방지를 위한 페널티
    eos_token_id=pipe.tokenizer.eos_token_id,
    pad_token_id=pipe.tokenizer.pad_token_id
)

# 3. 실행
outputs = pipe(prompt, generation_config=gen_config)

# 4. 출력
full_text = outputs[0]['generated_text']
answer = full_text.split("<start_of_turn>model\n")[-1].strip()
print(answer)

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

모델을 만들고 싶은 사람에게는 모델링에 대한 설명도 필요하다. 그래서 <시스템 디자인(System Design)> 책으로부터 모형화 방법론인 '아키텍처'와 관련된 내용들을 알려주었다.​그리고, 데이터 과학자들이 사용하는 언급할 수 있는 용어들로 공부하도록 하였다.(예: 데이터베이스 설계/구축)​​그렇다면 어떤 분야든 아무리 잘한다 해봐라!​사람들은 자신만큼 열심히 하는 게 없다고 생각하지 않는 것 같았는데... (실제 상황에서는 다르겠지만)​우리는 항상 새로운 것을 배우며 성장하고 있다!!


In [13]:
# 프롬프트 수정
# (유저 프롬프트는 우리가 사용하는 부분, 시스템 프롬프트는 모델에 전해주는 규칙을 말합니다)
# user_question 부분에 자신이 하고 싶은 말을 넣어보십시오.
user_question = "데이터 분석이 필요한 이유는?"
prompt = f"<start_of_turn>user\n{user_question}<end_of_turn>\n<start_of_turn>model\n"

# 3. 실행
outputs = pipe(prompt, generation_config=gen_config)

# 4. 출력
full_text = outputs[0]['generated_text']
answer = full_text.split("<start_of_turn>model\n")[-1].strip()
print(answer)

모델을 만드는데 어려운 점은? <end_of_turn>
2. 데이터를 수집하는 방법에 대해서 알아보자.<start_of_turn>
- 1) 기존의 시스템과 연동하여 자동으로 정보수신(예: 전화통계, 트래픽 통제 등)<br /> - 즉시 처리가 가능하도록 하는 것이 중요하다.(ex : 네트워크 장애 발생 후 재구성 시간 단축 및 서비스 정상 복원시간 감소 효율적 관리)​- 30초 내로 응답할 때까지 반응 속도 향샹​ ​[네이버]에서 제공되는 실험 결과입니다.​ [실행결론]- 대용량 DB와 함께 사용하면 더욱 빠르고 안정적인 검색 성능 구현.- 하루 평균 조회 건당 최대 약5만건 이상 고속검색 지원 (최저연산횟감=47회)- 최근 접촉 IP 중복률 제거 기능 적용해 클라이언트 PC 상황 변형 없음​​*기본적으로 스마일 서버에서는 지난번에도 설명 드렸지만 메인프레임 환경에는 다음 두 가지 문제점들이 있습니다.*첫째, 많은 양 자료들을 일괄처리하기 위해서는 그야말로 '빠른' 작업환경 조치해야 합니다.'스피디함', 그리고 '안전하고 신뢰받게 운영될 것'.둘쨰,'다양하게 연결된 다중 디바이스들'(PC/IPAD등), 각종 애플릿이나 웹브라우져들의 특징 때문에 한 번에 여러개 파일 또는 페이지 열기를 원활히 할지 못합니다.[그림6]<장비별 모니터링 화면><표8>.


적은 크기의 파라미터를 가지는 모델은 우리가 생각하는것 만큼 똑 부러지는 대답을 잘 못할 가능성이 큽니다.

 그래서 자신의 개인 컴퓨터, 로컬 환경에서 실무에 사용될 만한 답변을 얻으려면 파라미터가 큰 모델이 돌아갈 만한 GPU가 필요합니다.

대부분의 경우 집에 수백만원 하는 GPU가 없기 때문에, 기업의 GPU 자체를 할당받아 사용하는 방식 또는 모델의 출력값만 API방식으로 받아오는 방식 (closed model)을 사용합니다. 다음 편에서는 Open ai 라이브러리로 그럴듯한 답변을 가져오는 방식을 소개하겠습니다.

### 마치며

우리는 **인코더(Encoder)**를 통해 문장의 숨은 감정을 읽어내고, **디코더(Decoder)**를 통해 자연스러운 대화를 이어가는 인공지능의 사용법을 알아보았습니다.

 **허깅페이스(Hugging Face)**라는 거대한 보물창고를 통해 내 컴퓨터 안에서 인공지능을 부리는 법을 배운 것이죠.

인공지능은 멀리 있지 않습니다. 오늘 배운 pipeline 코드 몇 줄을 시작으로, 여러분의 업무 현장에 있는 수많은 텍스트 데이터를 분석하고 자동화하는 도전을 멈추지 마세요. 내 문제 상황에 딱 맞는 '최적의 모델'을 찾아 직접 적용해 보는 그 순간, 여러분은 단순한 사용자를 넘어 AI를 다루는 데이터 분석가로 거듭나게 될 것입니다.
